# Flow_Ai — Fine-Tune Qwen3-1.7B on Your Use Cases (Google Colab T4)

This notebook fine-tunes **Qwen/Qwen3-1.7B** with **Unsloth QLoRA**, keeping fast direct non-thinking responses (`enable_thinking=False`) and exporting a **Q4_K_M GGUF** ready to load into your local Ollama (`ollama create flowforge-copilot -f training/Modelfile.copilot`).

**Before running:**
1. Use the **Save to train** button inside **Chat with Qwen** (or edit `training/custom_examples.jsonl`) and run `node training/build_dataset.mjs`.
2. Set Colab Runtime → **Change runtime type** → **T4 GPU**.
3. Upload either:
   - **`copilot_train.jsonl` + `copilot_eval.jsonl`** (to train the **Chat with Qwen** bot specifically), OR
   - **`train.jsonl` + `eval.jsonl`** (to train both Copilot + Execution Node tasks).

In [ ]:
!nvidia-smi
!pip install -q unsloth trl datasets accelerate bitsandbytes

In [ ]:
from google.colab import files
print("Upload training/dist/train.jsonl and training/dist/eval.jsonl:")
uploaded = files.upload()

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 4096
MODEL_NAME = "unsloth/Qwen3-1.7B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
from datasets import load_dataset
import os

train_file = "copilot_train.jsonl" if os.path.exists("copilot_train.jsonl") else "train.jsonl"
eval_file = "copilot_eval.jsonl" if os.path.exists("copilot_eval.jsonl") else "eval.jsonl"
print(f"Using dataset: {train_file} / {eval_file}")

dataset = load_dataset("json", data_files={"train": train_file, "eval": eval_file})

def format_chat(example):
    # enable_thinking=False keeps Qwen3 outputs fast and direct (matching Flow_Ai think:false)
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return {"text": text}

train_ds = dataset["train"].map(format_chat)
eval_ds = dataset["eval"].map(format_chat)
print(train_ds[0]["text"][:500])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        eval_strategy="epoch",
        optim="adamw_8bit",
        output_dir="flowforge_lora",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer.train()

In [ ]:
# Quick sanity test before exporting
FastLanguageModel.for_inference(model)
test_messages = [
    {"role": "system", "content": "Return ONLY a valid JSON object matching the requested fields."},
    {"role": "user", "content": "Extract {\"service\": string, \"status\": string} from: Auth API responded healthy in 19ms."}
]
inputs = tokenizer.apply_chat_template(test_messages, tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=128, temperature=0.2)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# Export merged 4-bit GGUF (~1.3 GB) for local Ollama
model.save_pretrained_gguf(
    "flowforge_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("Exported files:")
!ls -lh flowforge_gguf/*.gguf